In [1]:
# Setup: Add project root to Python path
import sys
from pathlib import Path

# Notebook is in examples/ subdirectory, go up one level to project root
project_root = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"[OK] Added to Python path: {project_root}")
print(f"[OK] Can now import 'materials' package")

[OK] Added to Python path: c:\Users\user\Repo\Scripts\section_design_checks
[OK] Can now import 'materials' package


In [2]:
# Materials library imports
from materials.reinforced_concrete.materials import ConcreteMaterial, Rebar
from materials.reinforced_concrete.constitutive import ConcreteModelType, SteelModelType
from materials.reinforced_concrete.analysis import create_interaction_diagram
from materials.reinforced_concrete.code_checks.ec2_2004 import BendingCheck
from materials.reinforced_concrete.geometry import (
    create_rectangular_section,
    create_circular_section,
    create_linear_rebar_layer,
    create_rectangular_perimeter_rebars,
    RCSection,
)

In [3]:
# Create C30/37 concrete
concrete = ConcreteMaterial(grade="C30/37")

# Create B500B reinforcing steel
rebar_20 = Rebar(
    grade="B500B",
    diameter=20,      # mm
)

In [4]:
# Create rectangular section (centred at origin)
section_width = 300  # mm
section_height = 500  # mm

# Sections are created with their centre at (0,0)

section = create_rectangular_section(
    width=section_width,
    height=section_height,
    section_name="Rectangular Beam",
)

# ϕ20 => radius = 10
cover = 50
r = rebar_20.diameter / 2

x_left  = cover + r   # 50 + 10 = 60
x_right =  section_width - cover - r   # 300 - 60 = 240

y_bottom = cover + r  # 50 + 10 = 60
y_top    =  section_height - cover - r  # 500 - 60 = 440

bottom_layer = create_linear_rebar_layer(
    rebar=rebar_20,
    n_bars=3,
    start_point=(x_left,  y_bottom),
    end_point=(x_right, y_bottom),
    layer_name="bottom",
)
section.add_rebar_group(bottom_layer)

top_layer = create_linear_rebar_layer(
    rebar=rebar_20,
    n_bars=2,
    start_point=(x_left, y_top),
    end_point=(x_right, y_top),
    layer_name="top",
)
section.add_rebar_group(top_layer)

In [7]:
section.plot(
    concrete=concrete,
    show=False)


bending_check = BendingCheck(
    section=section,
    concrete=concrete,
    concrete_model_type=ConcreteModelType.PARABOLA_RECTANGLE,
    steel_model_type=SteelModelType.HORIZONTAL
)

moment = 163
axial = 0

results = bending_check.perform_check(
    M_Ed=moment,
    N_Ed=axial
)

results.__str__()

#print(f"Utilisation: {results.utilization:.2f}")

'Bending check (EC2 §6.1): WARNING (utilization: 99.1%) [N: 0.00/0.00 kN, M: 163.00/164.55 kN·m] - High utilization - consider increasing section or reinforcement'

In [8]:
# Create M-N interaction diagram
diagram = create_interaction_diagram(
    section=section,
    concrete=concrete,
    concrete_model_type=ConcreteModelType.PARABOLA_RECTANGLE,  # EC2 Fig 3.3
    steel_model_type=SteelModelType.HORIZONTAL,                  # With strain hardening
    n_fibres_width=20,
    n_fibres_height=30,
)

loadcases = []

forces_dict = {
    "M_Ed": moment,
    "N_Ed": axial
    }

loadcases.append(forces_dict)

diagram.plot_mn(
    load_points=loadcases,
    show_vectors=True,
    show=True
    )

diagram.plot_stress_strain(
    M_Ed=moment,
    N_Ed=axial,
    height=800,
    show=False,
    section_render="filled")